<a href="https://colab.research.google.com/github/elhamod/IS883_Fall_2026/blob/main/Session%2004/IS883_Session04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS883 Session 4: Prompting, Reasoning, and Hallucinations

Last week you called the Gemini API and learned to control it with the **system instruction** and a few generation parameters. This week we focus on the single most important skill for getting good results out of an LLM: **how you ask**.

We will cover:
1. **Prompt structure** — separating fixed instructions from the changing input.
2. **Few-shot prompting** — teaching the model by example.
3. **Chain-of-Thought** — asking the model to reason step by step.
4. **Structured output** — getting data instead of prose.
5. **Hallucinations** — why models confidently make things up, and how to limit it.

# Part 0: Setup

In [ ]:
# Install Google's GenAI SDK for the Gemini API (same package as Session 3).
!pip install -U google-genai

**Where the API key comes from.** Never paste a key into a notebook — anyone you share the file with gets your key, and a key pushed to a public repo is found within minutes. Instead the helper below looks for it in three places, in order:

1. **Colab Secrets** (the 🔑 panel on the left) — this is how you should do it in this course.
2. An **environment variable** of the same name — for VS Code or a local Jupyter.
3. Failing both, it **asks you to paste it**, and even then the value never lands in the file.

In [ ]:
# Read the API key from wherever this notebook is running, without writing it into the file.
import os, getpass

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass.getpass(f"{name}: ")

In [ ]:
# Connect to the Gemini API using the key you saved as a Colab secret (named MyGeminiKey).
from google import genai
from google.genai import types
import time

client = genai.Client(api_key=get_key('MyGeminiKey'))

MODEL = "gemini-3.1-flash-lite"

In [ ]:
# A small helper that sends one prompt to Gemini and returns the text answer.
# It is the Session 3 helper, trimmed to what we need this week.
def ask_gemini(prompt, system_instruction=None, temperature=None, response_schema=None):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=temperature,
        # Only ask for JSON when we pass a schema (used in the structured-output section).
        response_mime_type="application/json" if response_schema else None,
        response_schema=response_schema,
        # Keep the model's built-in "thinking" minimal so we can see the effect of OUR prompts.
        thinking_config=types.ThinkingConfig(thinking_level="MINIMAL", include_thoughts=False),
    )
    for attempt in range(4):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt, config=config).text
        except Exception as error:
            # Two things stop a call that is otherwise fine: the free tier's per-minute
            # limit (429), and the shared model being briefly overloaded (503). Both pass.
            if any(code in str(error) for code in ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE")):
                print("   (API busy, waiting 15 seconds...)")
                time.sleep(15)
            else:
                raise
    return "[gave up after repeated retries]"

# Part 1: Prompt Structure

A good prompt usually has two parts that we keep separate:
- **Fixed instructions** — the part that is the same for every user (the "template").
- **The changing input** — the specific text this particular user gave us.

Keeping them separate lets you reuse and improve the instructions without rewriting them each time. In Python, the simplest "template" is just an f-string with blanks.

In [ ]:
# A prompt template: fixed instructions with blanks ({}) that we fill in for each user.
def translation_prompt(text, language):
    return f"Translate the following text into {language}. Text: {text}"

# The instructions stay the same; only the blanks change from one user to the next.
print(ask_gemini(translation_prompt("My name is Mohannad", "Hindi")))
print(ask_gemini(translation_prompt("Where is the train station?", "Spanish")))

Remember the difference from Session 3: the **prompt** is the user's request, while the **system instruction** is the model's standing order that applies no matter what the user asks. Below, the same request produces a very different answer once we set a persona.

In [ ]:
# Same user request, but the system instruction changes the model's whole behavior.
request = "Where is the train station?"
print("No persona:", ask_gemini(request))
print("Conductor:", ask_gemini(request,
      system_instruction="You are a terse train conductor. Answer in one short sentence."))

**Experiment.** Change `translation_prompt` to also fix the *tone* (e.g., "translate politely and formally"). Notice you only edit the template in one place, and every call improves.

# Part 2: Few-Shot Prompting

Sometimes plain instructions are not enough — especially when you need the model to follow *your* convention that it has no way to guess. The fix is **few-shot prompting**: show the model a few worked examples inside the prompt, and it copies the pattern.

- **Zero-shot** = instructions only, no examples.
- **Few-shot** = a handful of example inputs and their correct outputs, then the real input.

Below, we route customer messages into an internal category list. Zero-shot, the model invents its own category names. Few-shot, it uses ours.

In [ ]:
# Route one customer message into our support team's four internal labels, two ways.
message = "The app logged me out and now my password won't work."

# Zero-shot: an instruction and nothing else.
zero_shot = f"Categorize this customer message in one or two words: {message}"
print("Zero-shot:", ask_gemini(zero_shot))

# Few-shot: three completed examples first, then the real message with the label left blank.
few_shot = f"""Categorize each customer message using ONLY these labels: BILLING, LOGIN, BUG, OTHER.

Message: I was charged twice this month.
Label: BILLING

Message: The screen freezes when I tap the map.
Label: BUG

Message: I can't remember which email I signed up with.
Label: LOGIN

Message: {message}
Label:"""
print("Few-shot: ", ask_gemini(few_shot))

**Question 1.** Why does the few-shot version reliably produce one of *your* four labels while the zero-shot version does not? *(5 points)*


**Answer**

*Provide your answer here*

# Part 3: Chain-of-Thought (CoT)

For questions that need a few logical steps, forcing the model to answer instantly often gives a wrong or inconsistent answer. **Chain-of-Thought** prompting asks the model to *reason step by step first*, which usually makes the final answer more reliable.

We will test this with a small physics riddle: does an object float in a liquid? (The rule: something floats if it is *less dense* than the liquid.)

In [ ]:
# A question that needs a couple of reasoning steps to get right.
obj = "avocado"
liquid = "rubbing alcohol"
riddle = f"Would a {obj} float in {liquid}?"

In [ ]:
# Case A: force an instant one-word answer, with NO room to reason. Five runs.
for i in range(5):
    answer = ask_gemini(riddle, system_instruction="Answer with a single word: yes or no. No explanation.")
    print(i + 1, answer)

In [ ]:
# Case B: let the model reason first, THEN pull out the final yes/no in a second call.
reasoning = ask_gemini(f"{riddle} Think step by step before answering.")
print("Reasoning:\n", reasoning)

final = ask_gemini(f"Based on the reasoning below, answer with only 'yes' or 'no'.\n\n{reasoning}")
print("\nFinal answer:", final)

In [ ]:
# Case C: hand the model the exact steps to follow (usually the most reliable).
guided = f"""Decide whether a {obj} would float in {liquid} by following these steps:
1. State the approximate density of {obj}.
2. State the approximate density of {liquid}.
3. If the object's density is lower than the liquid's, it floats; otherwise it sinks.
4. Give the final answer as 'yes' or 'no'."""
print(ask_gemini(guided))

**Experiment.** Try other pairs by changing `obj` and `liquid` in the first cell of this part, e.g. `obj = "ice cube"` with `liquid = "water"`, or a stone in oil. Do the reasoning versions still win?

**Question 2.** Compare Cases A, B, and C. Which gave the most consistent answers across runs, and why do you think reasoning first helps? *(5 points)*

**Answer**

*Provide your answer here*

# Part 4: Structured Output

In Session 3 you saw that `response_schema` forces the model to return data in a shape your program can read. That is not just for tidy answers — it is what lets you turn *messy free text* into clean records. Here we extract trip details from a rambling paragraph into fixed fields.

We will do it **twice**: first by simply asking the model to fill in blanks, then by requiring a schema. The two look equally good on screen; only one of them is safe to build on.

In [ ]:
# Attempt 1: no schema — we just ask the model to fill in the blanks. Three runs.
essay = """On Veterans Day I woke up in Boston and decided to take a break. I went on Expedia
and bought a ticket to Houston to see my family. I flew JetBlue, arrived safely, and had a
great long weekend. It was the best $250 I ever spent -- though they did lose my luggage!"""

fill_in_the_blanks = f"""Read the text below and fill in the blanks.

Source:
Destination:
Airline:
Price in USD:

Text: {essay}"""

for i in range(3):
    print("--- run", i + 1, "---")
    print(ask_gemini(fill_in_the_blanks))

The *content* is usually right. The **format** is not ours to control: across runs the model may add a preamble, bold the labels, write `$250`, `250.00` or `250 USD`, or drop a field it thinks is missing. Our program, though, needs one number it can do arithmetic with.

In [ ]:
# To use that answer in code we have to go hunting for the price ourselves.
answer = ask_gemini(fill_in_the_blanks)
price = None
for line in answer.splitlines():
    if "Price" in line:
        price = line
print("What our code found:", price)
print("Its type:", type(price).__name__)

Two problems. It is still a **string**, not a number — `price * 2` would repeat the text rather than double the money. And the search only works while the model keeps writing the word "Price"; nothing in the prompt *requires* that. Every hallucinated space, currency symbol or re-worded label is a bug waiting for the day you are not watching.

Now the same extraction with a schema.

In [ ]:
# Attempt 2: describe the exact "form" we want filled in. Optional fields may be left blank.
from pydantic import BaseModel
from typing import Optional
import json

class TripDetails(BaseModel):
    source: str
    destination: str
    airline: Optional[str] = None
    price_usd: Optional[float] = None

# Same text, same request — but requiring a schema guarantees valid JSON we can use as data.
raw = ask_gemini(f"Extract the trip details from this text:\n\n{essay}", response_schema=TripDetails)
print(raw)

trip = json.loads(raw)
print("\nThe traveller paid $", trip["price_usd"], "to fly to", trip["destination"])

Same model, same text, same request — the difference is entirely in what we *required*.

| | Attempt 1: fill in the blanks | Attempt 2: `response_schema` |
|---|---|---|
| Field names | whatever the model writes | exactly `source`, `destination`, `airline`, `price_usd` |
| Missing values | may be omitted, or invented to fill the blank | `null` — the field is still there |
| `price_usd` | a string you must clean up | a real `float` you can compute with |
| Parsing code | string surgery you maintain forever | one `json.loads` |

The lesson generalizes past this cell: **when the output feeds a program rather than a person, asking politely is not the same as constraining.** Prompt wording is a request; a schema is a contract.

**Experiment.** Add a `duration_of_stay_days` field to `TripDetails` and re-run. The model fills in what it can infer and leaves the rest blank.

# Part 5: Hallucinations and "AI Slop"

A **hallucination** is when the model states something false with total confidence. It happens because an LLM predicts *plausible-sounding* text, not *verified* text — it has no built-in sense of "I don't actually know this." When this low-quality, confidently-wrong output piles up, people call it **"AI slop."** In a business setting, slop is dangerous: a made-up refund policy or a fake citation can cost real money and trust.

First, let's make the model hallucinate on purpose.

In [ ]:
# Ask about a paper that does not exist. Watch the model confidently invent details.
fake = ("Summarize the main findings of the 2021 research paper "
        "'Quantum Widgets for Retail Forecasting' by Alvarez and Chen.")
print(ask_gemini(fake))

### Mitigation 1: Give the model permission to say "I don't know"
Models often guess because the prompt implicitly demands an answer. Explicitly allowing "I don't know" reduces made-up content.

In [ ]:
# Mitigation 1: explicitly allow the model to decline instead of guessing.
guarded = ("If you do not have reliable information, reply exactly: "
           "'I don't have reliable information on that.' Do not guess.\n\n" + fake)
print(ask_gemini(guarded))

### Mitigation 2: Ground the model in a trusted source
The most reliable fix is to *give the model the facts* and tell it to use only those. This is the core idea behind Retrieval-Augmented Generation (RAG), which we build next week.

Below we hand the model a tiny **library**: excerpts from the abstracts of three papers that really exist. Then we ask it two things — one it cannot answer from the library, and one it can.

In [ ]:
# Mitigation 2: give the model a small library of real sources and restrict it to them.
# These are verbatim sentences from three real papers (arXiv IDs included so you can check them).
library = """
[1] DeepAR: Probabilistic Forecasting with Autoregressive Recurrent Networks.
Salinas, Flunkert, Gasthaus (2017), arXiv:1704.04110.
"In retail businesses, for example, forecasting demand is crucial for having the right
inventory available at the right time at the right place." "In this paper we propose DeepAR,
a methodology for producing accurate probabilistic forecasts, based on training an auto
regressive recurrent network model on a large number of related time series." "We show
through extensive empirical evaluation on several real-world forecasting data sets accuracy
improvements of around 15% compared to state-of-the-art methods."

[2] Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting.
Lim, Arik, Loeff, Pfister (2019), arXiv:1912.09363.
"In this paper, we introduce the Temporal Fusion Transformer (TFT) - a novel attention-based
architecture which combines high-performance multi-horizon forecasting with interpretable
insights into temporal dynamics."

[3] Quantum Machine Learning. Biamonte, Wittek, Pancotti, Rebentrost, Wiebe, Lloyd (2017),
arXiv:1611.09347.
"The field of quantum machine learning explores how to devise and implement concrete quantum
software that offers such advantages." "Recent work has made clear that the hardware and
software challenges are still considerable but has also opened paths towards solutions."
"""

def ask_library(question):
    """Ask a question, but allow only the library above as evidence."""
    return ask_gemini(f"Answer using ONLY the library below. If the library does not contain "
                      f"the answer, say so plainly and do not guess.\n"
                      f"{library}\nQuestion: {question}")

# The same fake paper as before - but now the model has real sources to check against.
print(ask_library(fake))

The library talks about retail forecasting *and* about quantum machine learning, so a paper called "Quantum Widgets for Retail Forecasting" looks like it belongs. It does not. Grounding turned a confident fabrication into an honest "not in these sources" — the model can now tell the difference because there is something to check.

Grounding does the other half of the job too: it makes answers *better*, not just safer. Here is a specific number the model has to get exactly right.

In [ ]:
# The same question asked two ways: from memory, then from the library.
detail = "How much accuracy improvement does DeepAR report, and compared to what?"

print("From memory:\n", ask_gemini(detail))
print("\nFrom the library:\n", ask_library(detail))

**Question 3.** In your own words, why is grounding (Mitigation 2) usually safer than just trusting the model's memory? Give one business example where a hallucination would be costly. *(5 points)*


**Answer**

*Provide your answer here*

# Summary

- **Prompt structure**: keep fixed instructions separate from the changing input.
- **Few-shot**: show examples to teach the model your convention.
- **Chain-of-Thought**: ask for step-by-step reasoning to improve hard answers.
- **Structured output**: asking for fields in the prompt gives you prose; `response_schema` gives you data.
- **Hallucinations**: allow "I don't know" and, best of all, ground the model in trusted sources.

Next week (**Session 5**) we give the model real **tools** (function calling) and real **knowledge** (retrieval + RAG).